In [0]:
%run "../00.common/01.environment-config.py"

In [0]:
# Databricks notebook source

# MAGIC %md
# MAGIC # Transform Sprints Data
# MAGIC
# MAGIC 1. Read Bronze `sprints` data
# MAGIC 2. Keep only the columns required for analytics
# MAGIC 3. Standardize column names
# MAGIC 4. Rename columns to meaningful names
# MAGIC 5. Validate business keys
# MAGIC 6. Remove duplicate records
# MAGIC 7. Transform `race_name` to Title Case
# MAGIC 8. Write the transformed data to Silver `sprints`

# COMMAND ----------

# MAGIC %run "../00.common/01.environment-config.py"

# COMMAND ----------

from pyspark.sql import functions as F

# COMMAND ----------

# ============================================================
# SOURCE AND TARGET PATHS
# ============================================================

bronze_path = f"{s3_root_path}/Bronze/Nisarga/sprints"

silver_path = f"{s3_root_path}/Silver/Nisarga/sprints"

print("Bronze Path :", bronze_path)
print("Silver Path :", silver_path)

# COMMAND ----------

# ============================================================
# STEP 1 - READ BRONZE SPRINTS DATA
# ============================================================

sprints_df = (
    spark.read
    .format("delta")
    .load(bronze_path)
)

display(sprints_df)

# COMMAND ----------

# ============================================================
# STEP 2 - SELECT REQUIRED COLUMNS
# ============================================================

sprints_selected_df = sprints_df.select(
    F.col("season"),
    F.col("round"),
    F.col("constructorId"),
    F.col("driverId"),
    F.col("date"),
    F.col("raceName"),
    F.col("grid"),
    F.col("laps"),
    F.col("number"),
    F.col("points"),
    F.col("position"),
    F.col("positionText"),
    F.col("status"),
    F.col("ingestion_timestamp"),
    F.col("source_file")
)

display(sprints_selected_df)

# COMMAND ----------

# ============================================================
# STEP 3 - STANDARDIZE COLUMN NAMES
# ============================================================

sprints_renamed_df = (
    sprints_selected_df
    .withColumnsRenamed({
        "constructorId": "constructor_id",
        "driverId": "driver_id",
        "raceName": "race_name",
        "date": "race_date",
        "grid": "grid_position",
        "laps": "completed_laps",
        "number": "car_number",
        "position": "final_position",
        "positionText": "final_position_text"
    })
)

display(sprints_renamed_df)

# COMMAND ----------

# ============================================================
# STEP 4 - BUSINESS KEY VALIDATION
# ============================================================

sprints_valid_df = (
    sprints_renamed_df
    .filter(
        F.col("season").isNotNull() &
        F.col("round").isNotNull() &
        F.col("constructor_id").isNotNull() &
        F.col("driver_id").isNotNull()
    )
)

display(sprints_valid_df)

# COMMAND ----------

# ============================================================
# STEP 5 - REMOVE DUPLICATES
# ============================================================

sprints_distinct_df = (
    sprints_valid_df
    .dropDuplicates([
        "season",
        "round",
        "constructor_id",
        "driver_id"
    ])
)

print(
    "Duplicate / Invalid Records Removed :",
    sprints_renamed_df.count() - sprints_distinct_df.count()
)

display(sprints_distinct_df)

# COMMAND ----------

# ============================================================
# STEP 6 - STANDARDIZE TEXT VALUES
# ============================================================

sprints_final_df = (
    sprints_distinct_df
    .withColumn(
        "race_name",
        F.initcap(F.trim(F.col("race_name")))
    )
)

display(sprints_final_df)

# COMMAND ----------

# ============================================================
# STEP 7 - WRITE TO SILVER
# ============================================================

(
    sprints_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_path)
)

print("Silver Sprints data successfully written to:")
print(silver_path)

# COMMAND ----------

# ============================================================
# STEP 8 - VALIDATE SILVER DATA
# ============================================================

silver_sprints_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)

display(silver_sprints_df)

print("Bronze Sprints Records :", sprints_df.count())
print("Silver Sprints Records :", silver_sprints_df.count())

# COMMAND ----------

# ============================================================
# STEP 9 - DISPLAY FINAL SCHEMA
# ============================================================

silver_sprints_df.printSchema()